# Baseline NSL_KDD Without Feature Augmentation — DNN, Five-Seed Evaluation

In [1]:
# Global reproducibility setup
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42
RANDOM_STATE = SEED
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("Global seed set to:", SEED)

Global seed set to: 42


In [6]:
from pathlib import Path
import gc
import json
import platform
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    multilabel_confusion_matrix,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [7]:
# ============================================================
# Load the pre-generated NSL-KDD augmented partitions
# ============================================================
from pathlib import Path

AUGMENTED_DATA_DIRECTORY = (
    Path.cwd()
    / "/kaggle/input/datasets/sofonyasz/nsl-kdd-manuscript"
)

TRAIN_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "NSL_KDD_Augmented_Train.csv"
)

VALIDATION_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "NSL_KDD_Augmented_Validation.csv"
)

TEST_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "NSL_KDD_Augmented_Test.csv"
)

for file_path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    TEST_FILE,
]:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Required augmented partition not found: {file_path.resolve()}"
        )

train_data = pd.read_csv(TRAIN_FILE)
validation_data = pd.read_csv(VALIDATION_FILE)
test_data = pd.read_csv(TEST_FILE)

# Retained for compatibility with the existing experiment metadata.
INPUT_CSV = AUGMENTED_DATA_DIRECTORY

print(f"Training data:   {train_data.shape}")
print(f"Validation data: {validation_data.shape}")
print(f"Test data:       {test_data.shape}")

display(train_data.head())


Training data:   (104258, 44)
Validation data: (14826, 44)
Test data:       (29433, 44)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,IF_Anomaly_Score,AE_Anomaly_Score,SUOD_Anomaly_Score,category
0,0.0,1.0,11.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,1.0,1.0,0.00,0.0,-0.409113,0.008093,-0.155286,DoS
1,0.0,1.0,49.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,1.00,1.0,-0.455302,0.095445,0.259301,DoS
2,0.0,1.0,20.0,9.0,172.0,0.0,0.0,0.0,0.0,0.0,...,0.18,0.00,0.4,0.0,0.02,0.0,-0.421444,0.086396,-0.270910,Normal
3,0.0,1.0,24.0,9.0,233.0,547.0,0.0,0.0,0.0,0.0,...,0.03,0.03,0.0,0.0,0.00,0.0,-0.361821,0.009765,-0.623136,Normal
4,0.0,1.0,24.0,9.0,330.0,453.0,0.0,0.0,0.0,0.0,...,0.01,0.02,0.0,0.0,0.00,0.0,-0.347595,0.004767,-0.797096,Normal


In [9]:
train_data.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,IF_Anomaly_Score,AE_Anomaly_Score,SUOD_Anomaly_Score,category
0,0.0,1.0,11.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,1.0,1.0,0.00,0.0,-0.409113,0.008093,-0.155286,DoS
1,0.0,1.0,49.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.0,0.0,1.00,1.0,-0.455302,0.095445,0.259301,DoS
2,0.0,1.0,20.0,9.0,172.0,0.0,0.0,0.0,0.0,0.0,...,0.18,0.00,0.4,0.0,0.02,0.0,-0.421444,0.086396,-0.270910,Normal
3,0.0,1.0,24.0,9.0,233.0,547.0,0.0,0.0,0.0,0.0,...,0.03,0.03,0.0,0.0,0.00,0.0,-0.361821,0.009765,-0.623136,Normal
4,0.0,1.0,24.0,9.0,330.0,453.0,0.0,0.0,0.0,0.0,...,0.01,0.02,0.0,0.0,0.00,0.0,-0.347595,0.004767,-0.797096,Normal


In [10]:
nsl_kdd_data.shape

(148517, 41)

## 2. Feature-target separation and stratified data splitting

In [10]:
## 2. Feature-target separation using the pre-split partitions

LABEL_COLUMN = "category"

anomaly_score_columns = [
    "IF_Anomaly_Score",
    "AE_Anomaly_Score",
    "SUOD_Anomaly_Score",
]

for dataset_name, dataset in {
    "Training": train_data,
    "Validation": validation_data,
    "Test": test_data,
}.items():
    if LABEL_COLUMN not in dataset.columns:
        raise KeyError(
            f"'{LABEL_COLUMN}' was not found in the {dataset_name} partition."
        )

    if dataset[LABEL_COLUMN].isna().any():
        raise ValueError(
            f"'{LABEL_COLUMN}' contains missing values in the "
            f"{dataset_name} partition."
        )

    missing_anomaly_features = [
        feature
        for feature in anomaly_score_columns
        if feature not in dataset.columns
    ]

    if missing_anomaly_features:
        raise KeyError(
            f"{dataset_name} partition is missing anomaly-score features: "
            f"{missing_anomaly_features}"
        )


# ============================================================
# Remove anomaly-score features for the No-FA experiment
# ============================================================
train_data = train_data.drop(
    columns=anomaly_score_columns
)

validation_data = validation_data.drop(
    columns=anomaly_score_columns
)

test_data = test_data.drop(
    columns=anomaly_score_columns
)


feature_columns = [
    column
    for column in train_data.columns
    if column != LABEL_COLUMN
]

if not feature_columns:
    raise ValueError(
        "No input features were found."
    )


for dataset_name, dataset in {
    "Validation": validation_data,
    "Test": test_data,
}.items():
    current_features = [
        column
        for column in dataset.columns
        if column != LABEL_COLUMN
    ]

    if current_features != feature_columns:
        raise ValueError(
            f"{dataset_name} feature columns/order do not match training."
        )


X_train = train_data[
    feature_columns
].copy()

X_val = validation_data[
    feature_columns
].copy()

X_test = test_data[
    feature_columns
].copy()


y_train_original = (
    train_data[
        LABEL_COLUMN
    ]
    .astype(str)
    .str.strip()
)

y_val_original = (
    validation_data[
        LABEL_COLUMN
    ]
    .astype(str)
    .str.strip()
)

y_test_original = (
    test_data[
        LABEL_COLUMN
    ]
    .astype(str)
    .str.strip()
)


for dataset_name, labels in {
    "Training": y_train_original,
    "Validation": y_val_original,
    "Test": y_test_original,
}.items():
    if labels.eq("").any():
        raise ValueError(
            f"'{LABEL_COLUMN}' contains empty labels "
            f"in the {dataset_name} partition."
        )


for dataset_name, feature_data in {
    "Training": X_train,
    "Validation": X_val,
    "Test": X_test,
}.items():
    non_numeric_columns = feature_data.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    if non_numeric_columns:
        raise TypeError(
            f"{dataset_name} contains non-numeric input features: "
            f"{non_numeric_columns}"
        )

    if not np.isfinite(
        feature_data.to_numpy(dtype=np.float32)
    ).all():
        raise ValueError(
            f"{dataset_name} input features contain missing "
            "or infinite values."
        )


print(f"Training feature matrix:   {X_train.shape}")
print(f"Validation feature matrix: {X_val.shape}")
print(f"Test feature matrix:       {X_test.shape}")


Training feature matrix:   (104258, 40)
Validation feature matrix: (14826, 40)
Test feature matrix:       (29433, 40)


In [11]:
# Use the pre-generated train/validation/test partitions directly.

# Preserve row positions within each partition for reproducibility artifacts.
train_indices = X_train.index.to_numpy(copy=True)
validation_indices = X_val.index.to_numpy(copy=True)
test_indices = X_test.index.to_numpy(copy=True)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train_original = y_train_original.reset_index(drop=True)
y_val_original = y_val_original.reset_index(drop=True)
y_test_original = y_test_original.reset_index(drop=True)

# Fit label encoding on training labels only and reuse the mapping.
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_original)

training_classes = set(label_encoder.classes_)
unknown_validation_classes = set(y_val_original.unique()) - training_classes
unknown_test_classes = set(y_test_original.unique()) - training_classes

if unknown_validation_classes or unknown_test_classes:
    raise ValueError(
        "Validation/test partitions contain classes absent from training."
    )

y_val = label_encoder.transform(y_val_original)
y_test = label_encoder.transform(y_test_original)
number_of_classes = len(label_encoder.classes_)

# Retain the original baseline variable name.
num_classes = number_of_classes

print("Multiclass label mapping:")
for class_code, class_name in enumerate(label_encoder.classes_):
    print(f"{class_code}: {class_name}")

expected_nsl_kdd_classes = {
    "normal",
    "dos",
    "probe",
    "r2l",
    "u2r",
}
observed_nsl_kdd_classes = {
    str(class_name).strip().lower()
    for class_name in label_encoder.classes_
}

if num_classes != 5 or observed_nsl_kdd_classes != expected_nsl_kdd_classes:
    raise ValueError(
        "Expected the five NSL-KDD categories "
        "Normal, DoS, Probe, R2L, and U2R; observed "
        f"{label_encoder.classes_.tolist()}."
    )

partition_sizes = np.array(
    [len(X_train), len(X_val), len(X_test)],
    dtype=int,
)
observed_split_fractions = partition_sizes / partition_sizes.sum()
expected_split_fractions = np.array(
    [0.702, 0.298 * (1.0 - 0.665), 0.298 * 0.665]
)
split_tolerance = 2.0 / partition_sizes.sum()

if not np.allclose(
    observed_split_fractions,
    expected_split_fractions,
    atol=split_tolerance,
    rtol=0.0,
):
    raise ValueError(
        "Unexpected NSL-KDD split proportions: "
        f"{observed_split_fractions.tolist()}"
    )

print(
    "Observed train/validation/test fractions: "
    f"{observed_split_fractions[0]:.6f}, "
    f"{observed_split_fractions[1]:.6f}, "
    f"{observed_split_fractions[2]:.6f}"
)

del train_data, validation_data, test_data


Multiclass label mapping:
0: DoS
1: Normal
2: Probe
3: R2L
4: U2R
Observed train/validation/test fractions: 0.701994, 0.099827, 0.198179


In [12]:
print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")
print(f"Number of training classes: {len(np.unique(y_train))}")

Training set:   (104258, 40)
Validation set: (14826, 40)
Test set:       (29433, 40)
Number of training classes: 5


In [13]:
def display_class_distribution(encoded_labels, partition_name):
    distribution = (
        pd.Series(encoded_labels)
        .value_counts(normalize=True)
        .sort_index()
        .rename(index=dict(enumerate(label_encoder.classes_)))
        .apply(lambda value: f"{value:.2%}")
    )
    print(f"\n{partition_name}:")
    print(distribution)

print("Class distribution (normalized frequency):")
display_class_distribution(y_train, "Training set")
display_class_distribution(y_val, "Validation set")
display_class_distribution(y_test, "Test set")

Class distribution (normalized frequency):

Training set:
DoS       35.95%
Normal    51.88%
Probe      9.48%
R2L        2.61%
U2R        0.08%
Name: proportion, dtype: object

Validation set:
DoS       35.95%
Normal    51.88%
Probe      9.48%
R2L        2.61%
U2R        0.08%
Name: proportion, dtype: object

Test set:
DoS       35.95%
Normal    51.88%
Probe      9.48%
R2L        2.61%
U2R        0.08%
Name: proportion, dtype: object


In [15]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape ,X_val.shape,y_val.shape

((104258, 40), (29433, 40), (104258,), (29433,), (14826, 40), (14826,))

In [16]:
y_train_count = (
    pd.Series(y_train).value_counts().sort_index()
    .rename(index=dict(enumerate(label_encoder.classes_)))
)
y_train_count

DoS       37477
Normal    54091
Probe      9882
R2L        2724
U2R          84
Name: count, dtype: int64

In [17]:
y_test_count = (
    pd.Series(y_test).value_counts().sort_index()
    .rename(index=dict(enumerate(label_encoder.classes_)))
)
y_test_count

DoS       10580
Normal    15271
Probe      2790
R2L         769
U2R          23
Name: count, dtype: int64

##### Validation-set class counts

In [18]:
y_val_count = (
    pd.Series(y_val).value_counts().sort_index()
    .rename(index=dict(enumerate(label_encoder.classes_)))
)
y_val_count

DoS       5330
Normal    7692
Probe     1405
R2L        387
U2R         12
Name: count, dtype: int64

### Standardize features after data splitting

In [19]:
# Fit only on training features, then apply the fitted scaler unchanged.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

for partition_name, values in {
    "training": X_train_scaled,
    "validation": X_val_scaled,
    "test": X_test_scaled,
}.items():
    if not np.isfinite(values).all():
        raise ValueError(f"Scaled {partition_name} features are invalid.")

print(f"Scaled training features:   {X_train_scaled.shape}")
print(f"Scaled validation features: {X_val_scaled.shape}")
print(f"Scaled test features:       {X_test_scaled.shape}")

# The unscaled partitions are no longer required in memory.
del X_train, X_val, X_test

Scaled training features:   (104258, 40)
Scaled validation features: (14826, 40)
Scaled test features:       (29433, 40)


## 3. Five-run baseline configuration and reusable artifacts


In [21]:
# Fixed repeated-training and inference-timing protocol
RUN_SEEDS = [42, 43, 44, 45, 46]
MODEL_TYPE = "DNN"
N_WARMUP_RUNS = 3
N_BENCHMARK_RUNS = 5

# Fixed baseline settings from the uploaded source notebook.
BASELINE_HYPERPARAMETERS = {
    "hidden_units_1": 64,
    "hidden_units_2": 32,
    "dropout_rate": 0.2,
    "learning_rate": 0.001,
    "l2_regularization": 1e-4,
    "epochs": 100,
    "batch_size": 64,
    "shuffle": True,
    "early_stopping": {
        "monitor": "val_loss",
        "patience": 10,
        "min_delta": 0.0,
        "restore_best_weights": True,
    },
}

OUTPUT_DIRECTORY = Path.cwd() / "NSL_KDD_baseline_without_FA_dnn_5_seed_results"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIRECTORY = OUTPUT_DIRECTORY / "reusable_artifacts"
ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Save fixed preprocessing objects once.
joblib.dump(scaler, ARTIFACT_DIRECTORY / "standard_scaler.joblib")
joblib.dump(label_encoder, ARTIFACT_DIRECTORY / "label_encoder.joblib")
with (ARTIFACT_DIRECTORY / "feature_columns.json").open("w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)
np.savez_compressed(
    ARTIFACT_DIRECTORY / "split_indices.npz",
    train=train_indices,
    validation=validation_indices,
    test=test_indices,
)

experiment_config = {
    "model": MODEL_TYPE,
    "dataset_path": str(INPUT_CSV),
    "feature_augmentation": False,
    "optuna": False,
    "split_random_state": int(RANDOM_STATE),
    "split_proportions": {"training": 0.702, "validation": 0.09983, "test": 0.19817},
    "run_seeds": RUN_SEEDS,
    "hyperparameters": BASELINE_HYPERPARAMETERS,
    "inference_protocol": {
        "warmup_runs": N_WARMUP_RUNS,
        "measured_runs": N_BENCHMARK_RUNS,
    },
}
with (OUTPUT_DIRECTORY / "experiment_config.json").open("w", encoding="utf-8") as f:
    json.dump(experiment_config, f, indent=2)

print("Model:", MODEL_TYPE)
print("Run seeds:", RUN_SEEDS)
print("Output directory:", OUTPUT_DIRECTORY)


Model: DNN
Run seeds: [42, 43, 44, 45, 46]
Output directory: /kaggle/working/NSL_KDD_baseline_without_FA_dnn_5_seed_results


## 4. Reproducibility, metrics, timing, and reporting helpers


In [22]:
def set_run_seed(seed):
    """Reset model-training randomness for one independent run."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except (AttributeError, RuntimeError):
    print("TensorFlow deterministic-operation control is unavailable.")


def benchmark_full_test_prediction(model, X_input):
    """Time complete-test prediction using 3 warm-ups + 10 measured runs."""
    warmup_size = min(1024, len(X_input))
    for _ in range(N_WARMUP_RUNS):
        model.predict(X_input[:warmup_size], verbose=0)

    measured_times = []
    probabilities = None
    for _ in range(N_BENCHMARK_RUNS):
        start_time = time.perf_counter()
        probabilities = model.predict(X_input, verbose=0)
        measured_times.append(time.perf_counter() - start_time)

    mean_inference_time = float(np.mean(measured_times))
    return probabilities, {
        "Inference_Time_s": mean_inference_time,
        "Latency_ms_per_sample": mean_inference_time / len(X_input) * 1000.0,
        "Throughput_samples_per_s": len(X_input) / mean_inference_time,
    }


def calculate_fpr_details(y_true, y_pred, labels):
    matrices = multilabel_confusion_matrix(y_true, y_pred, labels=labels)
    rows = []
    for class_index, matrix in enumerate(matrices):
        tn, fp, fn, tp = matrix.ravel()
        denominator = fp + tn
        fpr = fp / denominator if denominator else 0.0
        rows.append({
            "Class_Index": int(class_index),
            "TP": int(tp),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "FPR": float(fpr),
            "Support": int((y_true == labels[class_index]).sum()),
        })
    frame = pd.DataFrame(rows)
    micro_denominator = frame["FP"].sum() + frame["TN"].sum()
    summary = {
        "FPR_macro": float(frame["FPR"].mean()),
        "FPR_micro_overall": float(
            frame["FP"].sum() / micro_denominator if micro_denominator else 0.0
        ),
        "FPR_weighted": float(np.average(frame["FPR"], weights=frame["Support"])),
    }
    return frame, summary


def calculate_test_metrics(y_true, y_pred, probabilities, labels):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Precision_weighted": precision_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    try:
        metrics["ROC_AUC_macro_OvR"] = roc_auc_score(
            y_true, probabilities, labels=labels, average="macro", multi_class="ovr"
        )
    except ValueError as error:
        metrics["ROC_AUC_macro_OvR"] = np.nan
        print(f"Macro ROC-AUC could not be calculated: {error}")

    fpr_details, fpr_summary = calculate_fpr_details(y_true, y_pred, labels)
    metrics.update(fpr_summary)
    return {key: float(value) for key, value in metrics.items()}, fpr_details


def format_mean_sd(mean_value, sd_value, decimals=6, percentage=False):
    scale = 100.0 if percentage else 1.0
    if pd.isna(mean_value):
        return "NA"
    if pd.isna(sd_value):
        return f"{mean_value * scale:.{decimals}f} ± NA"
    return f"{mean_value * scale:.{decimals}f} ± {sd_value * scale:.{decimals}f}"


def save_confusion_matrix_figure(matrix, output_stem):
    """Save report-style confusion matrix with both axes rotated 45 degrees."""
    plt.figure(figsize=(6.8, 5.5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
    )
    plt.xlabel("Predicted label", fontsize=11)
    plt.ylabel("True label", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.yticks(rotation=45, va="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{output_stem}.pdf", bbox_inches="tight")
    plt.savefig(f"{output_stem}.png", dpi=600, bbox_inches="tight")
    plt.close()
def save_loss_curve(history_data, output_stem, title=None):
    """Save training/validation loss curve as PDF and 600-dpi PNG."""
    if isinstance(history_data, pd.DataFrame):
        history_frame = history_data.copy()
    else:
        history_frame = pd.DataFrame(history_data)

    if "loss" not in history_frame.columns:
        raise KeyError("The training history does not contain a 'loss' column.")

    epochs = np.arange(1, len(history_frame) + 1)

    plt.figure(figsize=(6.8, 5.0))
    plt.plot(epochs, history_frame["loss"], label="Training Loss")

    if "val_loss" in history_frame.columns:
        plt.plot(epochs, history_frame["val_loss"], label="Validation Loss")

    plt.xlabel("Epoch", fontsize=11)
    plt.ylabel("Loss", fontsize=11)
    if title:
        plt.title(title, fontsize=11)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{output_stem}.pdf", bbox_inches="tight")
    plt.savefig(f"{output_stem}.png", dpi=600, bbox_inches="tight")
    plt.close()


TensorFlow deterministic operations enabled.


## 5. Fixed DNN baseline model


In [23]:
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

num_features = X_train_scaled.shape[1]


def build_final_dnn():
    parameters = BASELINE_HYPERPARAMETERS
    regularizer = regularizers.l2(parameters["l2_regularization"])
    model = Sequential(name="Baseline_DNN")
    model.add(Input(shape=(num_features,)))
    model.add(Dense(parameters["hidden_units_1"], activation="relu", kernel_regularizer=regularizer))
    model.add(Dropout(parameters["dropout_rate"]))
    model.add(Dense(parameters["hidden_units_2"], activation="relu", kernel_regularizer=regularizer))
    model.add(Dropout(parameters["dropout_rate"]))
    model.add(Dense(num_classes, activation="softmax", kernel_regularizer=regularizer))
    model.compile(
        optimizer=Adam(learning_rate=parameters["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def make_early_stopping():
    settings = BASELINE_HYPERPARAMETERS["early_stopping"]
    return EarlyStopping(
        monitor=settings["monitor"],
        patience=settings["patience"],
        min_delta=settings["min_delta"],
        restore_best_weights=settings["restore_best_weights"],
        verbose=1,
    )

X_train_input = X_train_scaled
X_val_input = X_val_scaled
X_test_input = X_test_scaled


## 6. Execute five independent training runs and save per-seed reports


In [24]:
labels = np.arange(num_classes)
run_records = []
per_class_records = []
macro_records = []
weighted_records = []
representative_candidates = []

for run_number, seed in enumerate(RUN_SEEDS, start=1):
    print("\n" + "=" * 72)
    print(f"{MODEL_TYPE}: run {run_number}/{len(RUN_SEEDS)}; seed={seed}")
    print("=" * 72)

    seed_directory = OUTPUT_DIRECTORY / f"{MODEL_TYPE}_Seed_{run_number}"
    seed_directory.mkdir(parents=True, exist_ok=True)

    tf.keras.backend.clear_session()
    set_run_seed(seed)
    model = build_final_dnn()

    training_start = time.perf_counter()
    history_object = model.fit(
        X_train_input,
        y_train,
        validation_data=(X_val_input, y_val),
        epochs=BASELINE_HYPERPARAMETERS["epochs"],
        batch_size=BASELINE_HYPERPARAMETERS["batch_size"],
        callbacks=[make_early_stopping()],
        shuffle=BASELINE_HYPERPARAMETERS["shuffle"],
        verbose=1,
    )
    training_time = time.perf_counter() - training_start

    validation_probabilities = model.predict(X_val_input, verbose=0)
    validation_predictions = np.argmax(validation_probabilities, axis=1)
    validation_macro_f1 = f1_score(
        y_val, validation_predictions, labels=labels, average="macro", zero_division=0
    )

    evaluation = model.evaluate(X_test_input, y_test, verbose=0, return_dict=True)
    test_probabilities, timing_metrics = benchmark_full_test_prediction(model, X_test_input)
    test_predictions = np.argmax(test_probabilities, axis=1)
    test_metrics, fpr_details = calculate_test_metrics(
        y_test, test_predictions, test_probabilities, labels
    )

    run_record = {
        "Model": MODEL_TYPE,
        "Run": int(run_number),
        "Seed": int(seed),
        "Epochs_Trained": int(len(history_object.history["loss"])),
        "Training_Time_s": float(training_time),
        "Validation_F1_macro": float(validation_macro_f1),
        "Test_Loss": float(evaluation["loss"]),
        **test_metrics,
        **timing_metrics,
    }
    run_records.append(run_record)

    # Separate macro report for this run.
    macro_record = {
        "Model": MODEL_TYPE,
        "Run": int(run_number),
        "Seed": int(seed),
        "Precision_macro": run_record["Precision_macro"],
        "Recall_macro": run_record["Recall_macro"],
        "F1_macro": run_record["F1_macro"],
        "ROC_AUC_macro_OvR": run_record["ROC_AUC_macro_OvR"],
        "FPR_macro": run_record["FPR_macro"],
    }
    macro_records.append(macro_record)

    # Separate weighted-average report for this run.
    weighted_record = {
        "Model": MODEL_TYPE,
        "Run": int(run_number),
        "Seed": int(seed),
        "Precision_weighted": run_record["Precision_weighted"],
        "Recall_weighted": run_record["Recall_weighted"],
        "F1_weighted": run_record["F1_weighted"],
        "FPR_weighted": run_record["FPR_weighted"],
    }
    weighted_records.append(weighted_record)

    precision_values, recall_values, f1_values, support_values = precision_recall_fscore_support(
        y_test, test_predictions, labels=labels, zero_division=0
    )
    per_class_frame = fpr_details.copy()
    per_class_frame["Model"] = MODEL_TYPE
    per_class_frame["Run"] = int(run_number)
    per_class_frame["Seed"] = int(seed)
    per_class_frame["Class"] = label_encoder.classes_
    per_class_frame["Precision"] = precision_values
    per_class_frame["Recall"] = recall_values
    per_class_frame["F1"] = f1_values
    per_class_frame["Support"] = support_values.astype(int)
    per_class_frame = per_class_frame[[
        "Model", "Run", "Seed", "Class", "Precision", "Recall", "F1", "FPR",
        "TP", "TN", "FP", "FN", "Support"
    ]]
    per_class_records.extend(per_class_frame.to_dict(orient="records"))

    # Save per-seed scalar and separate macro/weighted reports.
    pd.DataFrame([run_record]).to_csv(seed_directory / "scalar_metrics.csv", index=False)
    pd.DataFrame([macro_record]).to_csv(seed_directory / "macro_metrics.csv", index=False)
    pd.DataFrame([weighted_record]).to_csv(seed_directory / "weighted_average_metrics.csv", index=False)
    per_class_frame.to_csv(seed_directory / "per_class_metrics_and_fpr.csv", index=False)

    fpr_summary_frame = pd.DataFrame([{
        "FPR_macro": run_record["FPR_macro"],
        "FPR_micro_overall": run_record["FPR_micro_overall"],
        "FPR_weighted": run_record["FPR_weighted"],
    }])
    fpr_summary_frame.to_csv(seed_directory / "false_positive_rate_summary.csv", index=False)

    report = classification_report(
        y_test,
        test_predictions,
        labels=labels,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(seed_directory / "classification_report.csv")

    matrix = confusion_matrix(y_test, test_predictions, labels=labels)
    pd.DataFrame(matrix, index=label_encoder.classes_, columns=label_encoder.classes_).to_csv(
        seed_directory / "confusion_matrix.csv"
    )
    save_confusion_matrix_figure(matrix, seed_directory / "confusion_matrix")

    history = {
        key: [float(value) for value in values]
        for key, values in history_object.history.items()
    }
    history_frame = pd.DataFrame(history)
    history_frame.to_csv(
        seed_directory / "training_history.csv",
        index=False,
    )
    save_loss_curve(
        history_frame,
        seed_directory / "loss_curve",
    )

    prediction_frame = pd.DataFrame({
        "True_Code": y_test,
        "True_Label": label_encoder.inverse_transform(y_test),
        "Predicted_Code": test_predictions,
        "Predicted_Label": label_encoder.inverse_transform(test_predictions),
    })
    for class_index, class_name in enumerate(label_encoder.classes_):
        prediction_frame[f"Probability_{class_name}"] = test_probabilities[:, class_index]
    prediction_frame.to_csv(seed_directory / "test_predictions_and_probabilities.csv", index=False)

    model_path = seed_directory / f"{MODEL_TYPE.lower()}_seed_{seed}.keras"
    model.save(model_path)

    run_metadata = {
        "Model": MODEL_TYPE,
        "Run": int(run_number),
        "Seed": int(seed),
        "Feature_Augmentation": False,
        "Optuna": False,
        "Split_Random_State": int(RANDOM_STATE),
        "Hyperparameters": BASELINE_HYPERPARAMETERS,
        "Inference_Warmups": N_WARMUP_RUNS,
        "Inference_Measured_Runs": N_BENCHMARK_RUNS,
    }
    with (seed_directory / "run_metadata.json").open("w", encoding="utf-8") as f:
        json.dump(run_metadata, f, indent=2)

    representative_candidates.append({
        "Seed": int(seed),
        "Run": int(run_number),
        "F1_macro": float(run_record["F1_macro"]),
        "Predictions": test_predictions.copy(),
        "Model_Path": str(model_path),
    })

    print(
        f"Completed {MODEL_TYPE} seed {seed}: "
        f"macro F1={run_record['F1_macro']:.6f}, "
        f"weighted F1={run_record['F1_weighted']:.6f}"
    )

    del model, history_object, validation_probabilities, test_probabilities
    gc.collect()
    tf.keras.backend.clear_session()

all_run_results = pd.DataFrame(run_records)
all_per_class_results = pd.DataFrame(per_class_records)
all_macro_results = pd.DataFrame(macro_records)
all_weighted_results = pd.DataFrame(weighted_records)

all_run_results.to_csv(OUTPUT_DIRECTORY / "dnn_5_seed_run_results.csv", index=False)
all_per_class_results.to_csv(OUTPUT_DIRECTORY / "dnn_per_class_5_seed_run_results.csv", index=False)
all_macro_results.to_csv(OUTPUT_DIRECTORY / "dnn_macro_metrics_5_seed_results.csv", index=False)
all_weighted_results.to_csv(OUTPUT_DIRECTORY / "dnn_weighted_average_metrics_5_seed_results.csv", index=False)

display(all_run_results)



DNN: run 1/5; seed=42
Epoch 1/100


2026-10-01 14:41:21.493070: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-10-01 14:41:21.664889: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


1630/1630 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9397 - loss: 0.2048 - val_accuracy: 0.9710 - val_loss: 0.0990
Epoch 2/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9677 - loss: 0.1129 - val_accuracy: 0.9775 - val_loss: 0.0844
Epoch 3/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9731 - loss: 0.0971 - val_accuracy: 0.9787 - val_loss: 0.0775
Epoch 4/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9770 - loss: 0.0870 - val_accuracy: 0.9829 - val_loss: 0.0712
Epoch 5/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9794 - loss: 0.0809 - val_accuracy: 0.9837 - val_loss: 0.0678
Epoch 6/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9803 - loss: 0.0774 - val_accuracy: 0.9848 - val_loss: 0.0670
Epoch 7/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9811 - loss: 0.0752 - val_accuracy: 0.9844 - val_loss: 0.0652
Epoch 8/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9814 - loss: 0.0727 - val_

2026-10-01 14:43:12.071305: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-01 14:43:17.094219: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Completed DNN seed 42: macro F1=0.869888, weighted F1=0.987907

DNN: run 2/5; seed=43
Epoch 1/100


...

2026-10-01 14:53:51.205671: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


1630/1630 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9390 - loss: 0.2098 - val_accuracy: 0.9713 - val_loss: 0.1029
Epoch 2/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9675 - loss: 0.1134 - val_accuracy: 0.9769 - val_loss: 0.0861
Epoch 3/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9730 - loss: 0.0973 - val_accuracy: 0.9793 - val_loss: 0.0787
Epoch 4/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9758 - loss: 0.0885 - val_accuracy: 0.9817 - val_loss: 0.0732
Epoch 5/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9770 - loss: 0.0849 - val_accuracy: 0.9833 - val_loss: 0.0701
Epoch 6/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9782 - loss: 0.0809 - val_accuracy: 0.9822 - val_loss: 0.0683
Epoch 7/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9794 - loss: 0.0772 - val_accuracy: 0.9827 - val_loss: 0.0667
Epoch 8/100
1630/1630 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9803 - loss: 0.0750 - val_

2026-10-01 14:57:17.097574: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-01 14:57:22.103316: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Completed DNN seed 46: macro F1=0.884323, weighted F1=0.989383


,Model,Run,Seed,Epochs_Trained,Training_Time_s,Validation_F1_macro,Test_Loss,Accuracy,Precision_macro,Recall_macro,...,Recall_weighted,F1_weighted,MCC,ROC_AUC_macro_OvR,FPR_macro,FPR_micro_overall,FPR_weighted,Inference_Time_s,Latency_ms_per_sample,Throughput_samples_per_s
0,DNN,1,42,34,110.467801,0.893210,0.057309,0.988007,0.909018,0.847508,...,0.988007,0.987907,0.979737,0.996243,0.003801,0.002998,0.007009,0.978815,0.033256,30070.028158
1,DNN,2,43,58,176.070791,0.886699,0.057650,0.987327,0.938143,0.837295,...,0.987327,0.987083,0.978564,0.995805,0.004198,0.003168,0.008316,0.972851,0.033053,30254.377822
2,DNN,3,44,76,213.974325,0.846002,0.055211,0.988822,0.972960,0.780760,...,0.988822,0.988471,0.981108,0.996049,0.003574,0.002794,0.006690,0.926347,0.031473,31773.192388
3,DNN,4,45,60,206.315836,0.887313,0.058543,0.987361,0.933272,0.846458,...,0.987361,0.987248,0.978653,0.995959,0.003996,0.003160,0.007343,1.018501,0.034604,28898.347806
4,DNN,5,46,66,210.229944,0.901172,0.054605,0.989468,0.906923,0.866913,...,0.989468,0.989383,0.982191,0.996851,0.003459,0.002633,0.006765,0.976436,0.033175,30143.308947


## 7. Five-run mean ± sample-SD summaries and representative results


In [26]:
identifier_columns = {"Model", "Run", "Seed"}
metric_columns = [column for column in all_run_results.columns if column not in identifier_columns]

summary_records = []
for metric_name in metric_columns:
    values = all_run_results[metric_name].astype(float)
    mean_value = float(values.mean())
    sd_value = float(values.std(ddof=1))
    summary_records.append({
        "Model": MODEL_TYPE,
        "Metric": metric_name,
        "Mean": mean_value,
        "SD": sd_value,
        "N": int(values.notna().sum()),
        "Mean ± SD": format_mean_sd(mean_value, sd_value),
    })

five_run_summary_long = pd.DataFrame(summary_records)
five_run_summary_long.to_csv(OUTPUT_DIRECTORY / "five_run_mean_sd_long.csv", index=False)
five_run_summary_long[["Metric", "Mean ± SD"]].to_csv(
    OUTPUT_DIRECTORY / "five_run_mean_sd_report_table.csv", index=False
)


def build_subset_summary(frame, metric_columns):
    rows = []
    for metric in metric_columns:
        values = frame[metric].astype(float)
        mean_value = float(values.mean())
        sd_value = float(values.std(ddof=1))
        rows.append({
            "Model": MODEL_TYPE,
            "Metric": metric,
            "Mean": mean_value,
            "SD": sd_value,
            "N": int(values.notna().sum()),
            "Mean ± SD": format_mean_sd(mean_value, sd_value),
        })
    return pd.DataFrame(rows)

macro_summary = build_subset_summary(
    all_macro_results,
    ["Precision_macro", "Recall_macro", "F1_macro", "ROC_AUC_macro_OvR", "FPR_macro"],
)
weighted_summary = build_subset_summary(
    all_weighted_results,
    ["Precision_weighted", "Recall_weighted", "F1_weighted", "FPR_weighted"],
)
macro_summary.to_csv(OUTPUT_DIRECTORY / "macro_metrics_5_seed_mean_sd.csv", index=False)
weighted_summary.to_csv(OUTPUT_DIRECTORY / "weighted_average_metrics_5_seed_mean_sd.csv", index=False)

# Compact publication tables. Accuracy is shown in both for convenient comparison.
accuracy_mean = float(all_run_results["Accuracy"].mean())
accuracy_sd = float(all_run_results["Accuracy"].std(ddof=1))

journal_macro = pd.DataFrame([{
    "Model": MODEL_TYPE,
    "Accuracy (%)": format_mean_sd(accuracy_mean, accuracy_sd, decimals=2, percentage=True),
    "Recall (macro %)": format_mean_sd(all_run_results["Recall_macro"].mean(), all_run_results["Recall_macro"].std(ddof=1), decimals=2, percentage=True),
    "Precision (macro %)": format_mean_sd(all_run_results["Precision_macro"].mean(), all_run_results["Precision_macro"].std(ddof=1), decimals=2, percentage=True),
    "F1-score (macro %)": format_mean_sd(all_run_results["F1_macro"].mean(), all_run_results["F1_macro"].std(ddof=1), decimals=2, percentage=True),
    "FPR (macro %)": format_mean_sd(all_run_results["FPR_macro"].mean(), all_run_results["FPR_macro"].std(ddof=1), decimals=4, percentage=True),
    "ROC-AUC (macro, OvR)": format_mean_sd(all_run_results["ROC_AUC_macro_OvR"].mean(), all_run_results["ROC_AUC_macro_OvR"].std(ddof=1), decimals=6),
}])

journal_weighted = pd.DataFrame([{
    "Model": MODEL_TYPE,
    "Accuracy (%)": format_mean_sd(accuracy_mean, accuracy_sd, decimals=2, percentage=True),
    "Recall (weighted %)": format_mean_sd(all_run_results["Recall_weighted"].mean(), all_run_results["Recall_weighted"].std(ddof=1), decimals=2, percentage=True),
    "Precision (weighted %)": format_mean_sd(all_run_results["Precision_weighted"].mean(), all_run_results["Precision_weighted"].std(ddof=1), decimals=2, percentage=True),
    "F1-score (weighted %)": format_mean_sd(all_run_results["F1_weighted"].mean(), all_run_results["F1_weighted"].std(ddof=1), decimals=2, percentage=True),
    "FPR (weighted %)": format_mean_sd(all_run_results["FPR_weighted"].mean(), all_run_results["FPR_weighted"].std(ddof=1), decimals=4, percentage=True),
}])

journal_macro.to_csv(OUTPUT_DIRECTORY / "journal_macro_mean_sd_table.csv", index=False)
journal_weighted.to_csv(OUTPUT_DIRECTORY / "journal_weighted_mean_sd_table.csv", index=False)

# Per-class five-run mean ± SD.
per_class_summary_records = []
for class_name, class_rows in all_per_class_results.groupby("Class", sort=False):
    record = {
        "Model": MODEL_TYPE,
        "Class": class_name,
        "Support": int(class_rows["Support"].iloc[0]),
        "N": int(len(class_rows)),
    }
    for metric in ["Precision", "Recall", "F1", "FPR"]:
        mean_value = float(class_rows[metric].mean())
        sd_value = float(class_rows[metric].std(ddof=1))
        record[f"{metric}_Mean"] = mean_value
        record[f"{metric}_SD"] = sd_value
        record[f"{metric}_Mean ± SD"] = format_mean_sd(mean_value, sd_value)
    per_class_summary_records.append(record)

per_class_mean_sd = pd.DataFrame(per_class_summary_records)
per_class_mean_sd.to_csv(OUTPUT_DIRECTORY / "per_class_five_run_mean_sd.csv", index=False)

# Representative run: macro F1 closest to the five-run mean.
mean_macro_f1 = float(all_run_results["F1_macro"].mean())
representative = min(
    representative_candidates,
    key=lambda item: abs(item["F1_macro"] - mean_macro_f1),
)
representative_seed = int(representative["Seed"])
representative_predictions = representative["Predictions"]
representative_matrix = confusion_matrix(y_test, representative_predictions, labels=labels)

pd.DataFrame(
    representative_matrix,
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
).to_csv(OUTPUT_DIRECTORY / "dnn_representative_confusion_matrix.csv")
save_confusion_matrix_figure(
    representative_matrix,
    OUTPUT_DIRECTORY / "dnn_representative_confusion_matrix",
)

representative_report = classification_report(
    y_test,
    representative_predictions,
    labels=labels,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(representative_report).transpose().to_csv(
    OUTPUT_DIRECTORY / "dnn_representative_classification_report.csv"
)

representative_runs = pd.DataFrame([{
    "Model": MODEL_TYPE,
    "Representative_Seed": representative_seed,
    "Representative_F1_macro": float(representative["F1_macro"]),
    "Five_Run_Mean_F1_macro": mean_macro_f1,
    "Model_Path": representative["Model_Path"],
}])
representative_runs.to_csv(OUTPUT_DIRECTORY / "representative_runs.csv", index=False)

representative_run_number = int(representative["Run"])
representative_seed_directory = (
    OUTPUT_DIRECTORY
    / f"{MODEL_TYPE}_Seed_{representative_run_number}"
)
representative_history_path = (
    representative_seed_directory / "training_history.csv"
)

if representative_history_path.exists():
    representative_history_frame = pd.read_csv(
        representative_history_path
    )
    representative_history_frame.to_csv(
        OUTPUT_DIRECTORY
        / "dnn_representative_training_history.csv",
        index=False,
    )
    save_loss_curve(
        representative_history_frame,
        OUTPUT_DIRECTORY
        / "dnn_representative_loss_curve",
    )
else:
    print(
        "Representative training history was not found:",
        representative_history_path,
    )


print("Five-run summary:")
display(five_run_summary_long)
print("Macro mean ± SD:")
display(macro_summary)
print("Weighted-average mean ± SD:")
display(weighted_summary)
print("Journal macro table:")
display(journal_macro)
print("Journal weighted table:")
display(journal_weighted)
print("Representative run:")
display(representative_runs)
print("All results saved to:", OUTPUT_DIRECTORY)


Five-run summary:


,Model,Metric,Mean,SD,N,Mean ± SD
0,DNN,Epochs_Trained,58.800000,15.530615,5,58.800000 ± 15.530615
1,DNN,Training_Time_s,183.411739,43.452828,5,183.411739 ± 43.452828
2,DNN,Validation_F1_macro,0.882879,0.021422,5,0.882879 ± 0.021422
3,DNN,Test_Loss,0.056664,0.001678,5,0.056664 ± 0.001678
4,DNN,Accuracy,0.988197,0.000935,5,0.988197 ± 0.000935
5,DNN,Precision_macro,0.932063,0.026806,5,0.932063 ± 0.026806
6,DNN,Recall_macro,0.835787,0.032597,5,0.835787 ± 0.032597
7,DNN,F1_macro,0.858060,0.038901,5,0.858060 ± 0.038901
8,DNN,Precision_weighted,0.988077,0.000972,5,0.988077 ± 0.000972
9,DNN,Recall_weighted,0.988197,0.000935,5,0.988197 ± 0.000935


Macro mean ± SD:


,Model,Metric,Mean,SD,N,Mean ± SD
0,DNN,Precision_macro,0.932063,0.026806,5,0.932063 ± 0.026806
1,DNN,Recall_macro,0.835787,0.032597,5,0.835787 ± 0.032597
2,DNN,F1_macro,0.858060,0.038901,5,0.858060 ± 0.038901
3,DNN,ROC_AUC_macro_OvR,0.996181,0.000407,5,0.996181 ± 0.000407
4,DNN,FPR_macro,0.003806,0.000302,5,0.003806 ± 0.000302


Weighted-average mean ± SD:


...

Journal weighted table:


,Model,Accuracy (%),Recall (weighted %),Precision (weighted %),F1-score (weighted %),FPR (weighted %)
0,DNN,98.82 ± 0.09,98.82 ± 0.09,98.81 ± 0.10,98.80 ± 0.09,0.7225 ± 0.0661


Representative run:


,Model,Representative_Seed,Representative_F1_macro,Five_Run_Mean_F1_macro,Model_Path
0,DNN,42,0.869888,0.85806,/kaggle/working/NSL_KDD_baseline_without_FA_dn...


All results saved to: /kaggle/working/NSL_KDD_baseline_without_FA_dnn_5_seed_results
